In [ ]:
import ipywidgets as widgets
from IPython.display import display
import threading, tempfile, os, sys, zipfile, requests, shutil, json, time
import subprocess, io
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from ipyleaflet import Map, GeoJSON, basemaps
from shapely.geometry import box, shape, mapping
from shapely.ops import unary_union
from IPython.display import Image as _IPyImage

os.environ['SHAPE_RESTORE_SHX'] = 'YES'
from shapely.strtree import STRtree
from datetime import date

import dashboard_utils.logging_utils as logging_utils
from dashboard_utils.logging_utils import log, WidgetStream
from dashboard_utils.hbv_utils import parse_predata, generate_predata_csv, find_mpirun
from dashboard_utils import api_client as _api
from dashboard_utils import downloader
from dashboard_utils.downloader import (
    fmi_s3_urls,
    download_and_merge_nc as _download_and_merge_nc,
    download_era5 as _download_era5,
    download_nc_url as _do_download_nc_url,
    download_shp as _do_download_shp,
    ensure_cdsapirc as _ensure_cdsapirc,
    FMI_VARIABLES, ERA5_VARIABLES,
    TASO_URLS, SYKE_URBAN_URL, SYKE_AGRI_URL,
)
from dashboard_utils.dem_utils import (
    _ensure_pysheds, _ensure_rasterio,
    path_to_rasterio as _path_to_rasterio,
    render_dem_overview as _render_dem_overview,
    hillshade as _hillshade,
    pixel_to_lonlat as _pixel_to_lonlat,
    render_to_png as _render_to_png,
)
from dashboard_utils.ui_styles import (
    inject_css, app_header_widget, render_stepper,
    section_lbl, divider, tip, right_panel_header,
)
inject_css()

# ═══════════════════════════════════════════════════════════════════════════
# Height constants — tune _CHROME to match your JupyterLab chrome height.
# JupyterLab 3/4 typical: menu(32) + toolbar(36) + tab-bar(32) + status(24) ≈ 124–148 px
# Increase _CHROME if the dashboard still overflows; decrease if there's a gap.
# ═══════════════════════════════════════════════════════════════════════════
_CHROME   = 148
_ROOT_PAD = 28
_HDR      = 58
_LTAB     = 36
_STEPPER  = 62
_NAV      = 52
_INNER    = _CHROME + _ROOT_PAD + _HDR + _LTAB + _STEPPER + _NAV

_PNL_H   = f'calc(100vh - {_CHROME + _ROOT_PAD}px)'
_STEP1_H = f'calc(100vh - {_INNER}px)'
_MAP_H   = f'calc(100vh - {_INNER + 70}px)'
_LOG_H   = f'calc(100vh - {_CHROME + _ROOT_PAD + 58 + 40 + 36}px)'

# ── Dark palette ─────────────────────────────────────────────────────────
_BG0 = '#020617'
_BG1 = '#0f172a'
_BG2 = '#1e293b'
_BDR = '#334155'
_BDR2 = '#1e293b'

# ── GeoJSON styles for dark map ──────────────────────────────────────────
_STYLE_DEFAULT  = {'color':'#60a5fa','fillColor':'#1e3a5f','fillOpacity':0.45,'weight':1.2}
_STYLE_HOVER    = {'fillColor':'#fb923c','fillOpacity':0.65}
_STYLE_SELECTED = {'color':'#fb923c','fillColor':'#7c2d12','fillOpacity':0.60,'weight':2.5}

# ── Dark placeholder helpers ─────────────────────────────────────────────
def _dark_placeholder(icon, title, sub=''):
    return widgets.HTML(
        f'<div style="width:100%;height:100%;background:{_BG1};'
        f'display:flex;align-items:center;justify-content:center;'
        f'flex-direction:column;gap:10px;padding:24px;box-sizing:border-box;">'
        f'<div style="font-size:42px;opacity:0.15">{icon}</div>'
        f'<div style="color:#334155;font-size:13px;font-weight:600;text-align:center">{title}</div>'
        + (f'<div style="color:#1e293b;font-size:11px;text-align:center">{sub}</div>' if sub else '')
        + '</div>',
        layout=widgets.Layout(width='100%', height='100%'),
    )

# ═══════════════════════════════════════════════════════════════════════════
# State
# ═══════════════════════════════════════════════════════════════════════════
state = {
    'temp_dir': tempfile.mkdtemp(), 'download_dir': None,
    'shapefile_path': None, 'predata_path': None, 'hbvpara_path': None,
    'output_files': {}, 'selected_id': None, 'selected_ids': set(),
    'id_col': None, 'gdf': None, 'gdf_wgs': None, 'all_ids': [],
    'highlight_layer': None, 'viewport_layer': None, 'leaflet_map': None,
    'last_bounds': None, 'current_taso': 'taso2', 'catchment_bounds': None,
    'precipitation_nc': None, 'evapotranspiration_nc': None, 'temperature_nc': None,
    'urban_land_path': None, 'agricultural_land_path': None,
    'sindex': None, 'id_to_idx': None,
    'shapefile_taso_key': None,
}

def _get_dl_dir():
    d = (state.get('download_dir') or '').strip()
    if d:
        try: os.makedirs(d, exist_ok=True); return d
        except Exception as e: log(f'Cannot create "{d}": {e} — using temp dir', 'warn')
    return state['temp_dir']

downloader.init(state, _get_dl_dir)
_MPIRUN = find_mpirun()
_WORKER_SCRIPT = os.path.join(os.getcwd(), 'hbv_worker.py')
_ensure_cdsapirc()
_USE_API = bool(os.environ.get('HBV_API_URL'))

# ═══════════════════════════════════════════════════════════════════════════
# ── STEP 1 widgets ─────────────────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════
taso_toggle = widgets.ToggleButtons(
    options=[('1','taso1'),('2','taso2'),('3','taso3'),('4','taso4'),('5','taso5')],
    value='taso2', description='', style={'description_width':'0px','button_width':'46px'},
)
taso_lbl = widgets.HTML('<small style="color:#475569">Taso 2</small>')
taso_toggle.observe(lambda c: setattr(taso_lbl, 'value',
    f'<small style="color:#475569">Taso {c["new"][-1]} selected</small>'), names='value')

shp_source_toggle = widgets.ToggleButtons(
    options=[('SYKE auto','syke'),('Upload own','upload')],
    value='syke', description='', style={'description_width':'0px','button_width':'110px'},
)
shp_load_btn  = widgets.Button(description='⬇ Load', button_style='info',
                                layout=widgets.Layout(width='90px'))
shp_auto_prog = widgets.IntProgress(value=0, min=0, max=100,
                                     layout=widgets.Layout(width='180px', visibility='hidden'))
shp_auto_lbl  = widgets.HTML('')
upload_shp    = widgets.FileUpload(
                              accept='.zip,.shp',
                              multiple=False,
                              description='Upload SHP',
                              layout=widgets.Layout(width='100%', visibility='hidden'))
upload_shp_lbl      = widgets.HTML('')
shp_pick_lbl        = widgets.HTML('')
upload_shp_load_btn = widgets.Button(description='Load', button_style='warning',
                                      layout=widgets.Layout(width='70px', visibility='hidden'))

def _on_shp_src(change):
    is_syke = change['new'] == 'syke'
    shp_load_btn.layout.visibility        = 'visible' if is_syke else 'hidden'
    upload_shp.layout.visibility          = 'hidden'  if is_syke else 'visible'
    upload_shp_load_btn.layout.visibility = 'hidden'  if is_syke else 'visible'
shp_source_toggle.observe(_on_shp_src, names='value')

def _load_syke(b):
    taso = taso_toggle.value; shp_load_btn.disabled = True
    state['shapefile_taso_key'] = taso
    shp_auto_prog.layout.visibility = 'visible'
    shp_auto_lbl.value = f'<small style="color:#475569">Checking HPC cache…</small>'
    log(f'Loading {taso}...', 'dl')
    threading.Thread(target=_dl_and_load, args=(TASO_URLS[taso], taso)).start()
shp_load_btn.on_click(_load_syke)

def _dl_and_load(url, taso):
    try:
        shp_auto_lbl.value = f'<small style="color:#475569">Downloading {taso}…</small>'
        fname = f'{taso}.zip'; dest = os.path.join(state['temp_dir'], fname)
        r = requests.get(url, stream=True, timeout=300); r.raise_for_status()
        total = int(r.headers.get('content-length', 0)); done = 0; _t = 0.0
        with open(dest, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk); done += len(chunk)
                    now = time.monotonic()
                    if now - _t >= 0.5:
                        shp_auto_prog.value = int(done/total*100) if total else 50
                        shp_auto_lbl.value = f'<small style="color:#475569">{done/1024/1024:.1f} MB</small>'
                        _t = now
        exdir = os.path.join(state['temp_dir'], f'shp_{taso}')
        os.makedirs(exdir, exist_ok=True)
        with zipfile.ZipFile(dest) as z: z.extractall(exdir)
        shps = [os.path.join(dp, fn) for dp, _, fns in os.walk(exdir) for fn in fns if fn.endswith('.shp')]
        match = [s for s in shps if taso in os.path.basename(s).lower()]
        path = match[0] if match else shps[0]
        shp_auto_lbl.value = '<small style="color:#475569">Loading…</small>'
        _load_shapefile_thread(path)
    except Exception as e:
        shp_auto_lbl.value = f'<small style="color:#ef4444">Error: {e}</small>'
        log(f'Download failed: {e}', 'error')
    finally:
        shp_load_btn.disabled = False

def _load_uploaded_shp(b):
    if not upload_shp.value:
        upload_shp_lbl.value = '<small style="color:#ef4444">Select a file first</small>'; return
    upload_shp_load_btn.disabled = True
    # FileUpload widget (ipywidgets 8): value is a tuple of dicts
    finfo = upload_shp.value[0]
    fname = finfo['name']
    content = bytes(finfo['content'])
    exdir = os.path.join(state['temp_dir'], 'shp_upload')
    shutil.rmtree(exdir, ignore_errors=True); os.makedirs(exdir, exist_ok=True)
    if fname.lower().endswith('.zip'):
        try:
            with zipfile.ZipFile(io.BytesIO(content)) as z: z.extractall(exdir)
            shps = [os.path.join(dp, fn) for dp, _, fns in os.walk(exdir)
                    for fn in fns if fn.lower().endswith('.shp')]
            if not shps:
                upload_shp_lbl.value = '<small style="color:#ef4444">No .shp in zip</small>'
                upload_shp_load_btn.disabled = False; return
            path = shps[0]
        except Exception as e:
            upload_shp_lbl.value = f'<small style="color:#ef4444">Zip: {e}</small>'
            upload_shp_load_btn.disabled = False; return
    elif fname.lower().endswith('.shp'):
        upload_shp_lbl.value = '<small style="color:#ef4444">Please upload a .zip containing all shapefile components (.shp .shx .dbf .prj)</small>'
        upload_shp_load_btn.disabled = False; return
    else:
        upload_shp_lbl.value = '<small style="color:#ef4444">.zip only (containing .shp + .shx + .dbf + .prj)</small>'
        upload_shp_load_btn.disabled = False; return
    upload_shp_lbl.value = '<small style="color:#475569">Loading…</small>'
    threading.Thread(target=lambda: (
        _load_shapefile_thread(path),
        upload_shp_load_btn.__setattr__('disabled', False)
    )).start()
upload_shp_load_btn.on_click(_load_uploaded_shp)

# ── Map container — dark placeholder until shapefile loads ─────────────────
map_container = widgets.VBox(
    [_dark_placeholder('🗺', 'Map will appear here',
                        'Load a shapefile using SYKE auto-download or upload your own')],
    layout=widgets.Layout(height=_MAP_H, min_height='300px', width='100%',
                           background_color=_BG1),
)
viewport_lbl = widgets.HTML(
    f'<small style="color:{_BDR};padding:4px 8px">—</small>'
)

MAX_FEATURES = {5:50,6:100,7:200,8:300,9:400,10:500,11:600,12:800,13:1000,14:1500}
SIMPLIFY_TOL = {5:0.02,6:0.01,7:0.005,8:0.002,9:0.001,10:0.0005,11:0.0002,12:0.0001,13:0.00005,14:0.0}
_vp_lock = threading.Lock(); _vp_timer = [None]

def _refresh_viewport(bounds, zoom):
    if not _vp_lock.acquire(blocking=False): return
    try:
        gdf_wgs = state.get('gdf_wgs'); sindex = state.get('sindex')
        m = state.get('leaflet_map'); id_col = state.get('id_col')
        if gdf_wgs is None or m is None or sindex is None: return
        zoom = max(5, min(int(zoom), 14))
        max_feat = MAX_FEATURES.get(zoom, 0)
        if max_feat == 0:
            viewport_lbl.value = f'<small style="color:#475569;padding:4px 8px">Zoom in to see catchments</small>'; return
        s, w = bounds[0]; n, e = bounds[1]
        candidate_idx = sindex.query(box(w, s, e, n))
        if len(candidate_idx) == 0:
            viewport_lbl.value = f'<small style="color:#475569;padding:4px 8px">No catchments in view</small>'; return
        tol = SIMPLIFY_TOL.get(zoom, 0.001)
        in_view = gdf_wgs.iloc[candidate_idx[:max_feat]].copy()
        if tol > 0: in_view['geometry'] = in_view['geometry'].simplify(tol, preserve_topology=True)
        features = [{'type':'Feature','geometry':r['geometry'].__geo_interface__,
                     'properties':{id_col: str(r[id_col])}} for _, r in in_view.iterrows()]
        if state.get('viewport_layer') and state['viewport_layer'] in m.layers:
            m.remove_layer(state['viewport_layer'])
        def _on_click(event=None, feature=None, **kw):
            if feature:
                cid = feature.get('properties', {}).get(id_col)
                if cid: _highlight(cid)
        lyr = GeoJSON(data={'type':'FeatureCollection','features':features},
                      style=_STYLE_DEFAULT, hover_style=_STYLE_HOVER)
        lyr.on_click(_on_click); m.add_layer(lyr); state['viewport_layer'] = lyr
        viewport_lbl.value = (f'<small style="color:#475569;padding:4px 8px">'
                              f'{len(in_view)} catchments · zoom {zoom}</small>')
    except Exception as e:
        viewport_lbl.value = f'<small style="color:#f59e0b;padding:4px 8px">⚠ {e}</small>'
    finally:
        _vp_lock.release()

def _schedule_viewport(bounds, zoom):
    if _vp_timer[0] is not None: _vp_timer[0].cancel()
    t = threading.Timer(0.3, _refresh_viewport, args=(bounds, zoom))
    _vp_timer[0] = t; t.start()

catchment_status = widgets.HTML(
    '<div class="hbv-status-warn"><b>No catchment selected</b> — click a polygon on the map or search.</div>'
)

def _rebuild_highlight_layer():
    gdf_wgs = state.get('gdf_wgs'); id_col = state.get('id_col')
    id_to_idx = state.get('id_to_idx'); m = state.get('leaflet_map')
    if m is None or gdf_wgs is None or id_to_idx is None: return
    if state['highlight_layer'] and state['highlight_layer'] in m.layers:
        m.remove_layer(state['highlight_layer']); state['highlight_layer'] = None
    selected = state['selected_ids']
    if not selected: state['catchment_bounds'] = None; _refresh_selection_ui(); return
    features = [{'type':'Feature',
                 'geometry':gdf_wgs.iloc[id_to_idx[str(s)]]['geometry'].__geo_interface__,
                 'properties':{id_col: str(s)}}
                for s in selected if id_to_idx.get(str(s)) is not None]
    if not features: return
    hl = GeoJSON(data={'type':'FeatureCollection','features':features}, style=_STYLE_SELECTED)
    m.add_layer(hl); state['highlight_layer'] = hl
    rows = [gdf_wgs.iloc[id_to_idx[str(s)]] for s in selected if id_to_idx.get(str(s)) is not None]
    all_b = [r['geometry'].bounds for r in rows]
    state['catchment_bounds'] = (min(b[0] for b in all_b), min(b[1] for b in all_b),
                                  max(b[2] for b in all_b), max(b[3] for b in all_b))
    n = len(selected); ids_str = ', '.join(str(s) for s in sorted(selected, key=str)[:4])
    if n > 4: ids_str += f' +{n-4} more'
    catchment_status.value = (
        f'<div class="hbv-status-ok"><b>{n} catchment{"s" if n>1 else ""} selected:</b> '
        f'<code style="background:#052e16;padding:1px 6px;border-radius:3px;color:#4ade80">{ids_str}</code></div>'
    )
    _refresh_selection_ui()

def _highlight(sid):
    gdf_wgs = state.get('gdf_wgs'); id_col = state.get('id_col')
    id_to_idx = state.get('id_to_idx')
    if gdf_wgs is None or id_to_idx is None: return
    sid_str = str(sid); idx = id_to_idx.get(sid_str)
    if idx is None: return
    orig_gdf = state.get('gdf')
    if orig_gdf is not None:
        nm = orig_gdf[orig_gdf[id_col].astype(str) == sid_str]
        native_id = nm[id_col].values[0] if not nm.empty else gdf_wgs.iloc[idx][id_col]
    else:
        native_id = gdf_wgs.iloc[idx][id_col]
    if sid_str in state['selected_ids']:
        state['selected_ids'].discard(sid_str); log(f'Deselected: {native_id}', 'map')
    else:
        state['selected_ids'].add(sid_str); state['selected_id'] = native_id
        log(f'Selected: {native_id} ({len(state["selected_ids"])} total)', 'map')
        row = gdf_wgs.iloc[idx]; b = row['geometry'].bounds
        state['leaflet_map'].fit_bounds([[b[1],b[0]],[b[3],b[2]]])
    _rebuild_highlight_layer(); _refresh_chip_grid()

def _clear_selection():
    state['selected_ids'].clear(); state['selected_id'] = None; state['catchment_bounds'] = None
    _rebuild_highlight_layer(); _refresh_chip_grid(); log('Selection cleared', 'map')

_sel_tags_html = widgets.HTML('')
_sel_count_lbl = widgets.HTML(
    f'<small style="color:{_BDR}">None selected</small>'
)
clear_sel_btn = widgets.Button(description='✕ Clear all', button_style='warning',
                                layout=widgets.Layout(width='100px', height='28px'))
clear_sel_btn.on_click(lambda b: _clear_selection())

_random_n_inp = widgets.BoundedIntText(value=60, min=1, max=500, step=1,
    description='', layout=widgets.Layout(width='70px'))
_random_sel_btn = widgets.Button(description='⚡ Select N random', button_style='info',
    layout=widgets.Layout(width='140px', height='28px'))

def _select_random(b):
    import random
    if state.get('gdf') is None:
        log('Load a shapefile first', 'warn'); return
    gdf = state['gdf']
    id_col = state.get('id_col', 'TASO_ID')
    all_ids = [str(v) for v in gdf[id_col].dropna().unique()]
    n = min(_random_n_inp.value, len(all_ids))
    chosen = random.sample(all_ids, n)
    state['selected_ids'] = set(chosen)
    _refresh_selection_ui()
    _update_readiness()
    log(f'Randomly selected {n} catchments', 'map')

_random_sel_btn.on_click(_select_random)

def _refresh_selection_ui():
    selected = state['selected_ids']
    if not selected:
        _sel_count_lbl.value = f'<small style="color:{_BDR}">None selected</small>'
        _sel_tags_html.value = ''; return
    _sel_count_lbl.value = f'<small style="color:#4ade80;font-weight:700">{len(selected)} selected</small>'
    _sel_tags_html.value = '<div style="margin:4px 0">' + ''.join(
        f'<span class="hbv-sel-tag">{s}</span>' for s in sorted(selected, key=str)) + '</div>'

# ── Unified search ─────────────────────────────────────────────────────────
search_unified   = widgets.Text(placeholder='🔍  Search by name, river or ID…',
                                 layout=widgets.Layout(width='100%'))
search_count_lbl = widgets.HTML(
    f'<small style="color:{_BDR}">Load a shapefile first</small>'
)
_chip_trigger    = widgets.Text(layout=widgets.Layout(display='none'))
_chip_trigger.observe(lambda c: _highlight(c['new'].strip()) if c['new'].strip() else None, names='value')
_chips_html      = widgets.HTML(
    f'<div class="hbv-chip-empty" style="color:{_BDR}">Load a shapefile to see catchments.</div>'
)
_current_chip_ids = []

def _refresh_chip_grid(query=None):
    ids = state.get('all_ids', [])
    sel = state.get('selected_ids', set())
    if not ids:
        _chips_html.value = f'<div class="hbv-chip-empty" style="color:{_BDR}">Load a shapefile to see catchments.</div>'
        return
    q = (query if query is not None else search_unified.value).strip().lower()
    if not q:
        display_ids = ids[:80]
        count_txt = f'Showing {len(display_ids)} of {len(ids)} — type to filter'
    else:
        gdf = state.get('gdf'); id_col = state.get('id_col')
        id_matches = [i for i in ids if q in i.lower()]
        name_matches = []
        if gdf is not None and id_col:
            for col in gdf.select_dtypes(include='object').columns:
                try:
                    hits = gdf[gdf[col].astype(str).str.lower().str.contains(q, na=False)]
                    for v in hits[id_col].values:
                        s = str(v)
                        if s not in id_matches: name_matches.append(s)
                except: pass
        display_ids = list(dict.fromkeys(id_matches + name_matches))[:80]
        count_txt = f'{len(display_ids)} result{"s" if len(display_ids)!=1 else ""}'
    _current_chip_ids[:] = display_ids
    search_count_lbl.value = f'<small style="color:#475569">{count_txt}</small>'
    model_id = _chip_trigger._model_id
    chips = []
    for cid in display_ids:
        cls  = 'hbv-chip sel' if str(cid) in sel else 'hbv-chip'
        safe = str(cid).replace("'", "\\'")
        js   = (f"(function(){{"
                f"var el=document.querySelector('[data-model-id=\"{model_id}\"] input');"
                f"if(el){{el.value='{safe}';el.dispatchEvent(new Event('change',{{bubbles:true}}));}}"
                f"}})()")
        chips.append(f'<span class="{cls}" onclick="{js}">{cid}</span>')
    _chips_html.value = (f'<div class="hbv-chip-grid">{"".join(chips)}</div>'
                         if chips else f'<div class="hbv-chip-empty" style="color:{_BDR}">No matches found.</div>')

search_unified.observe(lambda c: _refresh_chip_grid(c['new']), names='value')

def _load_shapefile_thread(path):
    try:
        gdf = gpd.read_file(path)
        geom_types = gdf.geometry.geom_type.value_counts().to_dict()
        log(f'Shapefile: {geom_types} | CRS: {gdf.crs}', 'map')
        if all(t in ('Point','MultiPoint') for t in geom_types):
            parent = os.path.dirname(path)
            for alt_path in [os.path.join(dp, fn) for dp,_,fns in os.walk(parent)
                             for fn in fns if fn.lower().endswith('.shp') and fn != os.path.basename(path)]:
                alt_gdf = gpd.read_file(alt_path)
                if any(t in ('Polygon','MultiPolygon') for t in alt_gdf.geometry.geom_type.values):
                    gdf = alt_gdf; path = alt_path
                    geom_types = gdf.geometry.geom_type.value_counts().to_dict(); break
            else:
                shp_pick_lbl.value = '<small style="color:#ef4444">Only point geometry</small>'; return
        id_col = next((c for c in gdf.columns if c.lower().endswith('_id')), None)
        if not id_col:
            shp_pick_lbl.value = f'<small style="color:#ef4444">No *_id column</small>'; return
        gdf_wgs = gdf[[id_col,'geometry']].copy().to_crs('EPSG:4326')
        gdf_wgs[id_col] = gdf_wgs[id_col].astype(str); gdf_wgs = gdf_wgs.reset_index(drop=True)
        sindex    = STRtree(gdf_wgs.geometry.values)
        id_to_idx = {str(v): i for i, v in enumerate(gdf_wgs[id_col])}
        state.update({'shapefile_path': path, 'gdf': gdf, 'gdf_wgs': gdf_wgs,
                      'id_col': id_col, 'all_ids': list(id_to_idx.keys()),
                      'sindex': sindex, 'id_to_idx': id_to_idx})
        b = gdf_wgs.total_bounds; center = [(b[1]+b[3])/2, (b[0]+b[2])/2]
        # Show loading state in map container
        map_container.children = [_dark_placeholder('⏳', 'Rendering map…', f'{len(gdf)} catchments loaded')]
        # CartoDB Dark Matter basemap
        m = Map(
            center=center, zoom=6,
            basemap=basemaps.CartoDB.DarkMatter,
            prefer_canvas=True,
            layout=widgets.Layout(width='100%', height='100%', min_height='300px'),
        )
        def _on_bounds(change):
            nb = change['new']
            if nb and nb != state['last_bounds']:
                state['last_bounds'] = nb; _schedule_viewport(nb, m.zoom)
        m.observe(_on_bounds, names='bounds')
        state.update({'leaflet_map': m, 'highlight_layer': None, 'viewport_layer': None})
        map_container.children = [m]   # replace placeholder with live map
        shp_auto_prog.layout.visibility = 'hidden'
        shp_pick_lbl.value = f'<small style="color:#4ade80">✔ {os.path.basename(path)} | {len(gdf)} catchments</small>'
        catchment_status.value = (
            '<div class="hbv-status-warn"><b>Shapefile loaded</b> — click a catchment on the map or search.</div>'
        )
        _refresh_chip_grid('')
        log(f'Loaded: {os.path.basename(path)} | {len(gdf)} catchments', 'ok')
    except Exception as e:
        import traceback
        map_container.children = [_dark_placeholder('⚠', f'Error: {e}', 'Check the log panel for details')]
        shp_pick_lbl.value = f'<small style="color:#ef4444">Error: {e}</small>'
        print(traceback.format_exc())

shp_pick_lbl = widgets.HTML('')

# ═══════════════════════════════════════════════════════════════════════════
# ── STEP 2 widgets ─────────────────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════
date_start = widgets.DatePicker(description='Start:', value=date(1991,1,1),
                                 layout=widgets.Layout(width='220px'))
date_end   = widgets.DatePicker(description='End:',   value=date(1991,12,31),
                                 layout=widgets.Layout(width='220px'))
date_lbl   = widgets.HTML('')
def _date_check(c):
    if date_start.value and date_end.value:
        date_lbl.value = ('<small style="color:#ef4444">⚠ End must be after start</small>'
                          if date_end.value <= date_start.value
                          else f'<small style="color:#4ade80">✔ {(date_end.value-date_start.value).days} days</small>')
date_start.observe(_date_check, names='value'); date_end.observe(_date_check, names='value')

_default_dl_dir = os.path.join(os.path.expanduser('~'), 'hbv_downloads')
state['download_dir'] = _default_dl_dir
_fb_state = {'cwd': os.path.expanduser('~')}
download_dir_input = widgets.Text(value=_default_dl_dir, description='Folder:',
                                   style={'description_width':'50px'}, layout=widgets.Layout(width='100%'))
_fb_listing  = widgets.HTML(''); _fb_status = widgets.HTML('')
_fb_new_name = widgets.Text(placeholder='new_folder_name', layout=widgets.Layout(width='200px'))
_fb_select_btn = widgets.Button(description='✔ Select', button_style='success',
                                 layout=widgets.Layout(width='100px'))
_fb_up_btn = widgets.Button(description='⬆ Up', layout=widgets.Layout(width='70px'))

def _fb_render():
    cwd = _fb_state['cwd']
    try:
        entries = sorted([e for e in os.scandir(cwd) if e.is_dir()], key=lambda e: e.name.lower())
        rows = ''.join(
            f'<tr><td style="padding:2px 8px"><a href="#" onclick="var el=document.getElementById(\'fb_click\');"'
            f'el.value=\'{e.path.replace(chr(39),"")}\';el.dispatchEvent(new Event(\'change\',{{bubbles:true}}));'
            f'return false;" style="color:#60a5fa;text-decoration:none;font-size:12px">📁 {e.name}</a></td></tr>'
            for e in entries[:60])
        suffix = f'<tr><td style="color:#475569;font-size:11px">… +{len(entries)-60}</td></tr>' if len(entries)>60 else ''
        _fb_listing.value = (
            f'<div style="background:{_BG1};border:1px solid {_BDR2};border-radius:6px;'
            f'padding:8px;max-height:140px;overflow-y:auto;">'
            f'<div style="color:#475569;font-size:11px;font-weight:600;margin-bottom:4px">📂 {cwd}</div>'
            f'<table style="width:100%">{rows}{suffix}</table></div>'
        )
    except PermissionError:
        _fb_listing.value = '<small style="color:#ef4444">Permission denied</small>'

_fb_click_input = widgets.Text(layout=widgets.Layout(display='none'))
_fb_click_input.observe(
    lambda c: (_fb_state.update({'cwd':c['new']}), _fb_new_name.__setattr__('value',''), _fb_render())
    if c['new'] else None, names='value')
_fb_up_btn.on_click(lambda b: (
    _fb_state.update({'cwd': os.path.dirname(_fb_state['cwd'])}), _fb_render())
    if os.path.dirname(_fb_state['cwd']) != _fb_state['cwd'] else None)
def _fb_select(b):
    chosen = os.path.join(_fb_state['cwd'], _fb_new_name.value.strip()) if _fb_new_name.value.strip() else _fb_state['cwd']
    try:
        os.makedirs(chosen, exist_ok=True); state['download_dir'] = chosen
        download_dir_input.value = chosen
        _fb_status.value = f'<small style="color:#4ade80">✔ {chosen}</small>'
        _fb_state['cwd'] = chosen; _fb_render()
    except Exception as e:
        _fb_status.value = f'<small style="color:#ef4444">Error: {e}</small>'
_fb_select_btn.on_click(_fb_select)
def _on_dl_dir_change(c):
    d = c['new'].strip()
    if d:
        state['download_dir'] = d
        _fb_state['cwd'] = d if os.path.isdir(d) else os.path.dirname(d) or os.path.expanduser('~')
        try: os.makedirs(d, exist_ok=True); _fb_status.value = '<small style="color:#4ade80">✔</small>'
        except Exception as e: _fb_status.value = f'<small style="color:#ef4444">{e}</small>'
        _fb_render()
    else:
        state['download_dir'] = None
        _fb_status.value = f'<small style="color:{_BDR}">using temp</small>'
download_dir_input.observe(_on_dl_dir_change, names='value')
_fb_render(); _on_dl_dir_change({'new': _default_dl_dir})

def _make_climate_panel(label_text, var_key, state_key):
    src = widgets.ToggleButtons(
        options=[('FMI S3','fmi'),('ERA5 API','era5'),('URL','custom'),('HPC Path','nfs')],
        value='fmi', description='', style={'description_width':'0px','button_width':'90px'},
    )
    folder, _, var_desc, *_ = FMI_VARIABLES[var_key]
    fmi_start = widgets.DatePicker(description='Start:', value=date(1991,1,1), layout=widgets.Layout(width='200px'))
    fmi_end   = widgets.DatePicker(description='End:',   value=date(1991,12,31), layout=widgets.Layout(width='200px'))
    fmi_date_lbl = widgets.HTML(''); fmi_url_out = widgets.HTML('')
    fmi_prog = widgets.IntProgress(value=0,min=0,max=100, layout=widgets.Layout(width='180px',visibility='hidden'))
    fmi_lbl  = widgets.HTML('')
    fmi_prev_btn = widgets.Button(description='Preview URLs', layout=widgets.Layout(width='120px'))
    fmi_dl_btn   = widgets.Button(description='⬇ Download', button_style='info', layout=widgets.Layout(width='120px'))
    copy_btn     = widgets.Button(description='Copy dates', layout=widgets.Layout(width='100px'))
    copy_btn.on_click(lambda b: (setattr(fmi_start,'value',date_start.value), setattr(fmi_end,'value',date_end.value)))
    def _fmi_dates(c):
        if fmi_start.value and fmi_end.value:
            fmi_date_lbl.value = ('<small style="color:#ef4444">End after start</small>'
                                  if fmi_end.value <= fmi_start.value
                                  else f'<small style="color:#475569">{(fmi_end.value-fmi_start.value).days}d ({fmi_end.value.year-fmi_start.value.year+1} files)</small>')
    fmi_start.observe(_fmi_dates, names='value'); fmi_end.observe(_fmi_dates, names='value')
    def _preview(b):
        urls = fmi_s3_urls(var_key, fmi_start.value, fmi_end.value)
        links = '<br>'.join(f'<a href="{u}" target="_blank" style="color:#60a5fa;font-size:11px">{os.path.basename(u)}</a>' for u in urls)
        fmi_url_out.value = f'<div style="background:{_BG1};border:1px solid {_BDR2};padding:6px;border-radius:6px;margin-top:4px">{links}</div>'
    fmi_prev_btn.on_click(_preview)
    fmi_dl_btn.on_click(lambda b: _download_and_merge_nc(var_key, state_key, fmi_start.value,
                                                          fmi_end.value, fmi_prog, fmi_lbl, fmi_dl_btn))
    fmi_panel = widgets.VBox([
        widgets.HTML(f'<small style="color:#475569">{var_desc}</small>'),
        widgets.HBox([fmi_start, fmi_end, fmi_date_lbl], layout=widgets.Layout(align_items='center', gap='6px')),
        widgets.HBox([copy_btn, fmi_prev_btn, fmi_dl_btn], layout=widgets.Layout(gap='6px', margin='4px 0')),
        widgets.HBox([fmi_prog, fmi_lbl], layout=widgets.Layout(align_items='center', gap='8px')),
        fmi_url_out,
    ])
    cds_var, _, _ = ERA5_VARIABLES[var_key]
    era5_start = widgets.DatePicker(description='Start:', value=date(1991,1,1), layout=widgets.Layout(width='200px'))
    era5_end   = widgets.DatePicker(description='End:',   value=date(1991,12,31), layout=widgets.Layout(width='200px'))
    era5_lbl = widgets.HTML('')
    era5_area_lbl = widgets.HTML(f'<small style="color:{_BDR}">Select catchment first</small>')
    era5_prog = widgets.IntProgress(value=0,min=0,max=100, layout=widgets.Layout(width='180px',visibility='hidden'))
    era5_dl_btn   = widgets.Button(description='⬇ Download ERA5', button_style='info', layout=widgets.Layout(width='150px'))
    era5_copy_btn = widgets.Button(description='Copy dates', layout=widgets.Layout(width='100px'))
    era5_copy_btn.on_click(lambda b: (setattr(era5_start,'value',date_start.value), setattr(era5_end,'value',date_end.value)))
    def _update_era5_area(*_):
        bv = state.get('catchment_bounds')
        era5_area_lbl.value = (
            f'<small style="color:#4ade80">N{bv[3]+.1:.2f} W{bv[0]-.1:.2f} S{bv[1]-.1:.2f} E{bv[2]+.1:.2f}</small>'
            if bv else '<small style="color:#ef4444">No catchment selected</small>')
    era5_dl_btn.on_click(lambda b: (_update_era5_area(),
                                     _download_era5(var_key, state_key, era5_start.value,
                                                    era5_end.value, era5_prog, era5_lbl, era5_dl_btn)))
    era5_panel = widgets.VBox([
        widgets.HTML(f'<small style="color:#475569">ERA5 — <code style="color:#93c5fd">{cds_var}</code></small>'),
        era5_area_lbl,
        widgets.HBox([era5_start, era5_end], layout=widgets.Layout(gap='6px')),
        widgets.HBox([era5_copy_btn, era5_dl_btn], layout=widgets.Layout(gap='6px', margin='4px 0')),
        widgets.HBox([era5_prog, era5_lbl], layout=widgets.Layout(align_items='center', gap='8px')),
    ])
    custom_url  = widgets.Text(placeholder=f'https://example.fi/{var_key}.nc',
                                description='URL:', layout=widgets.Layout(width='100%'))
    custom_btn  = widgets.Button(description='⬇ Download', button_style='info', layout=widgets.Layout(width='120px'))
    custom_prog = widgets.IntProgress(value=0,min=0,max=100, layout=widgets.Layout(width='180px',visibility='hidden'))
    custom_lbl  = widgets.HTML('')
    custom_btn.on_click(lambda b: (
        _do_download_nc_url(custom_url.value.strip(), var_key, state_key, custom_prog, custom_lbl, custom_btn)
        if custom_url.value.strip() else setattr(custom_lbl,'value','<small style="color:#ef4444">Enter URL</small>')))
    custom_panel = widgets.VBox([
        custom_url,
        widgets.HBox([custom_btn, custom_prog, custom_lbl],
                      layout=widgets.Layout(align_items='center', gap='8px', margin='4px 0')),
    ])
    nfs_path_inp = widgets.Text(
        placeholder='/data/hbv/uploads/<hash>/filename.nc  or  /data/hbv/shared/filename.nc',
        description='', layout=widgets.Layout(width='100%'))
    nfs_path_lbl = widgets.HTML('')
    def _on_nfs_path(c):
        p = c['new'].strip()
        if p:
            state[state_key] = p
            nfs_path_lbl.value = f'<small style="color:#4ade80">✔ path set</small>'
            _update_readiness()
        else:
            state[state_key] = None
            nfs_path_lbl.value = ''
            _update_readiness()
    nfs_path_inp.observe(_on_nfs_path, names='value')
    nfs_panel = widgets.VBox([
        widgets.HTML('<small style="color:#94a3b8">Paste the NFS path of a file already on the HPC (e.g. from a previous upload or scp). No upload needed.</small>'),
        nfs_path_inp,
        nfs_path_lbl,
    ])
    body = widgets.VBox([fmi_panel], layout=widgets.Layout(padding='8px 0'))
    def _on_src(c):
        if c['new'] == 'era5': _update_era5_area()
        body.children = ([fmi_panel] if c['new']=='fmi' else
                         [era5_panel] if c['new']=='era5' else
                         [nfs_panel] if c['new']=='nfs' else [custom_panel])
    src.observe(_on_src, names='value')
    return widgets.VBox([src, body])

precip_panel = _make_climate_panel('Precipitation',     'precipitation', 'precipitation_nc')
et_panel     = _make_climate_panel('Evapotranspiration','et',            'evapotranspiration_nc')
temp_panel   = _make_climate_panel('Temperature',        'temperature',   'temperature_nc')

# ═══════════════════════════════════════════════════════════════════════════
# ── STEP 3 widgets ─────────────────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════
def _make_land_panel(label, state_key, syke_url, syke_fname):
    status_lbl = widgets.HTML('')
    prog = widgets.IntProgress(value=0,min=0,max=100, layout=widgets.Layout(width='170px',visibility='hidden'))
    mode_tog = widgets.ToggleButtons(
        options=[('SYKE','syke'),('URL','url'),('Local','local'),('HPC Path','nfs')],
        value='syke', description='', style={'description_width':'0px','button_width':'80px'},
    )
    syke_btn = widgets.Button(description='⬇ Download', button_style='info', layout=widgets.Layout(width='120px'))
    syke_pnl = widgets.VBox([widgets.HTML(f'<small style="color:#475569;word-break:break-all">{syke_url}</small>'), syke_btn])
    url_inp = widgets.Text(placeholder='https://…', layout=widgets.Layout(width='100%'))
    url_btn = widgets.Button(description='⬇', button_style='info', layout=widgets.Layout(width='50px'))
    url_pnl = widgets.VBox([url_inp, url_btn])
    path_inp = widgets.FileUpload(accept='.zip,.shp', multiple=False, layout=widgets.Layout(width='100%'))
    path_btn = widgets.Button(description='Load', button_style='info', layout=widgets.Layout(width='70px'))
    path_pnl = widgets.VBox([path_inp, path_btn])
    nfs_land_inp = widgets.Text(
        placeholder='/data/hbv/uploads/<hash>/filename.shp  or  /data/hbv/shared/filename.shp',
        description='', layout=widgets.Layout(width='100%'))
    nfs_land_lbl = widgets.HTML('')
    def _on_nfs_land(c):
        p = c['new'].strip()
        if p:
            state[state_key] = p
            nfs_land_lbl.value = f'<small style="color:#4ade80">✔ path set</small>'
            status_lbl.value = f'<small style="color:#4ade80">✔ {os.path.basename(p)}</small>'
            _update_readiness()
        else:
            state[state_key] = None
            nfs_land_lbl.value = ''
            _update_readiness()
    nfs_land_inp.observe(_on_nfs_land, names='value')
    nfs_land_pnl = widgets.VBox([
        widgets.HTML('<small style="color:#94a3b8">Paste the NFS path of a file already on the HPC.</small>'),
        nfs_land_inp,
        nfs_land_lbl,
    ])
    body = widgets.VBox([syke_pnl], layout=widgets.Layout(padding='8px 0'))
    mode_tog.observe(
        lambda c: setattr(body, 'children',
                          [syke_pnl] if c['new']=='syke' else
                          [url_pnl] if c['new']=='url' else
                          [nfs_land_pnl] if c['new']=='nfs' else [path_pnl]),
        names='value')
    syke_btn.on_click(lambda b: threading.Thread(target=lambda: (
        _do_download_shp(syke_url, state_key, prog, status_lbl),
        syke_btn.__setattr__('disabled', False))).start() or setattr(syke_btn, 'disabled', True))
    url_btn.on_click(lambda b: (
        _do_download_shp(url_inp.value.strip(), state_key, prog, status_lbl)
        if url_inp.value.strip() else setattr(status_lbl,'value','<small style="color:#ef4444">Enter URL</small>')))
    def _load_local(b):
        if not path_inp.value:
            status_lbl.value = '<small style="color:#ef4444">Select a file first</small>'; return
        finfo = path_inp.value[0]
        fname = finfo['name']
        content = bytes(finfo['content'])
        exdir = os.path.join(state['temp_dir'], state_key+'_local')
        shutil.rmtree(exdir, ignore_errors=True); os.makedirs(exdir, exist_ok=True)
        if fname.lower().endswith('.zip'):
            with zipfile.ZipFile(io.BytesIO(content)) as z: z.extractall(exdir)
            shps = [os.path.join(dp, fn) for dp,_,fns in os.walk(exdir) for fn in fns if fn.endswith('.shp')]
            if not shps: status_lbl.value = '<small style="color:#ef4444">No .shp in zip</small>'; return
            state[state_key] = shps[0]
        elif fname.lower().endswith('.shp'):
            status_lbl.value = '<small style="color:#ef4444">Please upload a .zip containing all shapefile components</small>'; return
        else:
            status_lbl.value = '<small style="color:#ef4444">.zip only (containing .shp + .shx + .dbf + .prj)</small>'; return
        status_lbl.value = f'<small style="color:#4ade80">✔ {os.path.basename(state[state_key])}</small>'
        log(f'{label}: {state[state_key]}', 'ok')
    path_btn.on_click(_load_local)
    return widgets.VBox([mode_tog, body,
                          widgets.HBox([prog, status_lbl], layout=widgets.Layout(align_items='center', gap='8px'))])

urban_panel = _make_land_panel('Urban Land',        'urban_land_path',        SYKE_URBAN_URL, 'taajama')
agri_panel  = _make_land_panel('Agricultural Land', 'agricultural_land_path', SYKE_AGRI_URL,  'maatalousmaa2021')

# ═══════════════════════════════════════════════════════════════════════════
# ── STEP 4 widgets ─────────────────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════
upload_hbvpara = widgets.FileUpload(accept='.csv', multiple=False, layout=widgets.Layout(width='200px'))
lbl_hbvpara    = widgets.HTML(f'<small style="color:{_BDR}">No file</small>')

def _save_upload(state_key, label_w, change):
    val = change['new']
    if not val: return
    try:
        up = val[0] if isinstance(val, tuple) else list(val.values())[0]
        fname = up.get('name', f'{state_key}.csv'); content = bytes(up.get('content', b''))
        path  = os.path.join(state['temp_dir'], fname)
        with open(path, 'wb') as f: f.write(content)
        state[state_key] = path; label_w.value = f'<small style="color:#4ade80">✔ {fname}</small>'
    except Exception as e: label_w.value = f'<small style="color:#ef4444">{e}</small>'

upload_hbvpara.observe(
    lambda c: (_save_upload('hbvpara_path', lbl_hbvpara, c),
               log(f'hbv_para.csv: {os.path.basename(state["hbvpara_path"])}', 'ok')
               if state.get('hbvpara_path') else None), names='value')

output_freq = widgets.ToggleButtons(
    options=[('Daily','daily'),('Weekly','weekly'),('Monthly','monthly'),('Yearly','yearly')],
    value='daily', description='', style={'description_width':'0px','button_width':'80px'},
)
_max_cpu = 128  # HPC cluster max; updated from /cluster/info on load
cpu_slider = widgets.IntSlider(value=4, min=1, max=_max_cpu, step=1,
                                description='CPUs/node:', style={'description_width':'80px'},
                                layout=widgets.Layout(width='280px'))
n_nodes_slider = widgets.IntSlider(value=1, min=1, max=64, step=1,
                                    description='# nodes:', style={'description_width':'80px'},
                                    layout=widgets.Layout(width='280px'))
cluster_info_lbl = widgets.HTML('<small style="color:#94a3b8">Loading cluster info…</small>')
cluster_refresh_btn = widgets.Button(description='↻ Refresh', layout=widgets.Layout(width='90px'))
partition_dd = widgets.Dropdown(
    options=[('small (≤40 nodes, 4h)', 'small'), ('large (≤700 nodes, 3d)', 'large'), ('longrun (≤40 nodes, 14d)', 'longrun')],
    value='small', description='Partition:', style={'description_width': '70px'},
    layout=widgets.Layout(width='260px'),
)

def _refresh_cluster(b=None):
    try:
        ci = _api.get_cluster_info()
        s = ci.get('summary', {})
        nodes = ci.get('nodes', [])
        if 'error' in ci:
            cluster_info_lbl.value = f'<small style="color:#ef4444">⚠ {ci["error"]}</small>'
            return
        idle_n = s.get('idle_nodes', 0)
        total_n = s.get('total_nodes', 0)
        idle_c = s.get('idle_cpus', 0)
        alloc_c = s.get('alloc_cpus', 0)
        total_c = s.get('total_cpus', 0)
        # Update slider max to match real cluster
        if total_c > 0:
            max_cpu_per_node = max((nd['cpus_total'] for nd in nodes), default=4)
            cpu_slider.max = max_cpu_per_node
            cpu_slider.value = min(cpu_slider.value, max_cpu_per_node)
        if total_n > 0:
            n_nodes_slider.max = total_n
            n_nodes_slider.value = min(n_nodes_slider.value, idle_n or 1)
        # Update partition dropdown from live API data
        parts = ci.get('partitions', [])
        if parts:
            def _part_label(p):
                pct = int(100 * p['cpus_idle'] / p['cpus_total']) if p['cpus_total'] else 0
                color = '🟢' if pct > 30 else '🟡' if pct > 5 else '🔴'
                return f"{color} {p['name']} ({p['cpus_idle']}/{p['cpus_total']} CPUs idle, max {p['max_time']})"
            partition_dd.options = [(_part_label(p), p['name']) for p in parts]
            if partition_dd.value not in [p['name'] for p in parts]:
                partition_dd.value = parts[0]['name']
        node_color = '#4ade80' if idle_n > 0 else '#ef4444'
        cpu_color  = '#4ade80' if idle_c > 0 else '#ef4444'
        cluster_info_lbl.value = (
            f'<small>'
            f'<span style="color:{node_color}">⬡ {idle_n}/{total_n} nodes idle</span>'
            f' &nbsp; '
            f'<span style="color:{cpu_color}">⚙ {idle_c} idle / {alloc_c} busy / {total_c} total CPUs</span>'
            f'</small>'
        )
    except Exception as ex:
        cluster_info_lbl.value = f'<small style="color:#ef4444">⚠ cluster info: {ex}</small>'

cluster_refresh_btn.on_click(_refresh_cluster)
if _USE_API:
    import threading as _threading
    _threading.Thread(target=_refresh_cluster, daemon=True).start()

mpi_status_lbl = widgets.HTML(
    f'<small style="color:{"#4ade80" if _MPIRUN else "#94a3b8"}">'
    f'{"✔ mpirun found" if _MPIRUN else "ℹ mpirun not in local env (handled by cluster)"}</small>')
upload_predata  = widgets.FileUpload(accept='.csv', multiple=False, layout=widgets.Layout(width='200px'))
lbl_predata     = widgets.HTML(f'<small style="color:{_BDR}">Using UI inputs</small>')
fallback_toggle = widgets.ToggleButton(value=False, description='Use predata.csv override',
                                        layout=widgets.Layout(width='220px'))
fallback_box    = widgets.VBox(
    [widgets.HBox([upload_predata, lbl_predata], layout=widgets.Layout(align_items='center', gap='10px'))],
    layout=widgets.Layout(visibility='hidden'))
fallback_toggle.observe(
    lambda c: setattr(fallback_box.layout, 'visibility', 'visible' if c['new'] else 'hidden'), names='value')
upload_predata.observe(lambda c: _save_upload('predata_path', lbl_predata, c), names='value')
project_dir_input = widgets.Text(value='/home/jovyan/HBV-MODEL', description='Output:',
                                   style={'description_width':'50px'}, layout=widgets.Layout(width='100%'))
readiness_html = widgets.HTML('')
run_btn = widgets.Button(description='🚀  Run HBV Model', button_style='success',
                         layout=widgets.Layout(width='100%', height='46px'))
run_lbl = widgets.HTML(f'<small style="color:{_BDR}">Complete all steps first.</small>')

def _update_readiness():
    checks = [
        ('Catchment selected',  bool(state.get('selected_ids'))),
        ('Time range',          bool(date_start.value and date_end.value and date_end.value > date_start.value)),
        ('Precipitation data',  bool(state.get('precipitation_nc'))),
        ('Evapotranspiration',  bool(state.get('evapotranspiration_nc'))),
        ('Temperature data',    bool(state.get('temperature_nc'))),
        ('Urban land (SYKE)',   bool(state.get('urban_land_path'))),
        ('Agricultural land',   bool(state.get('agricultural_land_path'))),
        ('HBV parameters CSV',  bool(state.get('hbvpara_path'))),
    ]
    rows = ''; all_ok = True
    for name, ok in checks:
        icon = '✅' if ok else '⬜'; color = '#4ade80' if ok else '#475569'
        if not ok: all_ok = False
        rows += f'<div class="hbv-ready-row"><span style="font-size:14px">{icon}</span><span style="color:{color}">{name}</span></div>'
    status = ('<div style="margin-top:8px;font-size:12px;color:#4ade80;font-weight:700">✔ All ready — click Run!</div>'
              if all_ok else f'<div style="margin-top:8px;font-size:12px;color:{_BDR}">Complete missing items above.</div>')
    readiness_html.value = f'<div style="padding:4px 0">{rows}{status}</div>'
    run_btn.disabled = not all_ok

# ── Global API loading indicator ─────────────────────────────────────────────
_api_loading = widgets.HTML('', layout=widgets.Layout(height='3px'))
_api_calls   = [0]

def _api_start():
    _api_calls[0] += 1
    _api_loading.value = f'<div style="height:3px;background:linear-gradient(90deg,#3b82f6,#60a5fa,#3b82f6);background-size:200%;animation:shimmer 1.2s infinite"></div><style>@keyframes shimmer{{0%{{background-position:200%}}100%{{background-position:-200%}}}}</style>'

def _api_done():
    _api_calls[0] = max(0, _api_calls[0] - 1)
    if _api_calls[0] == 0:
        _api_loading.value = ''

# ── Right-panel log + output ────────────────────────────────────────────────
log_out = widgets.HTML('', layout=widgets.Layout(
    height=_LOG_H, min_height='200px',
    overflow_y='auto', padding='10px',
    background_color=_BG1,
))
logging_utils._log_out_ref[0] = log_out

clear_log_btn = widgets.Button(description='Clear', layout=widgets.Layout(width='70px', height='26px'))
clear_log_btn.on_click(lambda b: logging_utils.clear_log())

file_dd  = widgets.Dropdown(description='File:',   options=[], layout=widgets.Layout(width='100%'))
col_dd   = widgets.Dropdown(description='Column:', options=[], layout=widgets.Layout(width='100%'))
agg_tog  = widgets.ToggleButtons(options=['sum','mean'], value='sum',
                                  description='Agg:', style={'button_width':'70px'})
plot_btn = widgets.Button(description='Plot on Map', button_style='info', layout=widgets.Layout(width='140px'))
res_out  = widgets.Output()

right_log_tab = widgets.VBox([
    widgets.HBox([
        widgets.HTML(f'<span style="font-size:11px;font-weight:700;color:{_BDR};letter-spacing:0.5px">LIVE LOG</span>'),
        clear_log_btn,
    ], layout=widgets.Layout(justify_content='space-between', align_items='center',
                              padding='7px 12px', background_color=_BG0)),
    _api_loading,
    log_out,
])
right_output_tab = widgets.VBox([
    widgets.HTML('<div style="font-size:13px;font-weight:700;color:#e2e8f0;margin-bottom:8px">Model Results</div>'),
    file_dd, col_dd, agg_tog, plot_btn, res_out,
], layout=widgets.Layout(padding='12px', background_color=_BG2))

# ── Save results to JupyterHub workspace ──────────────────────────────────
def _save_to_workspace(job_id, local_results):
    """Copy downloaded CSVs into ~/hbv_results/<job_id>/ for persistent access."""
    workspace_dir = os.path.join(os.path.expanduser('~'), 'hbv_results', job_id[:8])
    try:
        os.makedirs(workspace_dir, exist_ok=True)
        count = 0
        for sid, paths in local_results.items():
            sid_dir = os.path.join(workspace_dir, sid)
            os.makedirs(sid_dir, exist_ok=True)
            for p in paths:
                if os.path.isfile(p):
                    shutil.copy2(p, os.path.join(sid_dir, os.path.basename(p)))
                    count += 1
        log(f'Saved {count} file(s) to ~/hbv_results/{job_id[:8]}/', 'ok')
    except Exception as exc:
        log(f'Workspace save error: {exc}', 'warn')

# ── Job History tab ───────────────────────────────────────────────────────
_jobs_rows_box = widgets.VBox([])   # updated in-place — no Output() needed
_jobs_status = widgets.HTML('')
_jobs_refresh_btn = widgets.Button(description='⟳ Refresh', button_style='',
                                    layout=widgets.Layout(width='90px'))

def _render_jobs():
    _api_start()
    _jobs_status.value = '<small style="color:#475569">Loading…</small>'
    try:
        jobs = _api.my_jobs()
    except Exception as exc:
        _jobs_status.value = f'<small style="color:#ef4444">Error: {exc}</small>'
        _api_done()
        return
    jobs = jobs[:20]  # show last 20 only — keeps widget fast
    _jobs_status.value = f'<small style="color:#475569">{len(jobs)} job(s) (last 20)</small>'
    if not jobs:
        _jobs_rows_box.children = [widgets.HTML('<div style="color:#64748b;font-size:12px;padding:8px">No jobs yet.</div>')]
        return
    # Render job list as a single lightweight HTML table + one Download button per done job
    table_rows = []
    dl_buttons  = []
    for j in jobs:
        jid = j['job_id']
        st  = j['status']
        col = '#4ade80' if st=='done' else '#ef4444' if st=='failed' else '#f59e0b' if st=='cancelled' else '#60a5fa'
        n   = len(j.get('catchments', []))
        ts  = (j.get('submitted_at') or '')[:16].replace('T', ' ')
        table_rows.append(
            f'<tr>'
            f'<td style="padding:4px 6px;color:#94a3b8;font-size:11px">{ts}</td>'
            f'<td style="padding:4px 6px;font-family:monospace;font-size:11px;color:#e2e8f0">{jid[:8]}…</td>'
            f'<td style="padding:4px 6px;font-size:11px"><span style="color:{col}">{st}</span></td>'
            f'<td style="padding:4px 6px;color:#64748b;font-size:11px">{n} catchments</td>'
            f'</tr>'
        )
        if st == 'done':
            btn = widgets.Button(description=f'⬇ {jid[:6]}', button_style='primary',
                                 layout=widgets.Layout(width='90px', height='26px'))
            def _make_dl(j=jid):
                def _dl(b):
                    b.disabled = True
                    threading.Thread(target=_download_old_job, args=(j,)).start()
                return _dl
            btn.on_click(_make_dl())
            dl_buttons.append(btn)
    table_html = widgets.HTML(
        f'<table style="width:100%;border-collapse:collapse">'
        f'<tr style="color:#475569;font-size:10px;border-bottom:1px solid #334155">'
        f'<th style="padding:3px 6px;text-align:left">Time</th>'
        f'<th style="padding:3px 6px;text-align:left">Job ID</th>'
        f'<th style="padding:3px 6px;text-align:left">Status</th>'
        f'<th style="padding:3px 6px;text-align:left">Size</th>'
        f'</tr>'
        + ''.join(table_rows) + '</table>'
    )
    children = [table_html]
    if dl_buttons:
        children.append(widgets.HTML('<div style="font-size:10px;color:#475569;padding:6px 6px 2px">Download results:</div>'))
        children.append(widgets.HBox(dl_buttons, layout=widgets.Layout(flex_wrap='wrap', gap='4px', padding='0 6px 8px')))
    _jobs_rows_box.children = tuple(children)
    _api_done()

def _on_jobs_refresh(b):
    threading.Thread(target=_render_jobs).start()
_jobs_refresh_btn.on_click(_on_jobs_refresh)

# Download handler — triggered by button click, runs full result download flow
_dl_job_lbl = widgets.HTML('')

def _download_old_job(job_id):
    _dl_job_lbl.value = f'<small style="color:#60a5fa">Downloading {job_id[:8]}…</small>'
    try:
        res = _api.get_results(job_id)
        remote_files = res.get('files', {})
        if not remote_files:
            _dl_job_lbl.value = '<small style="color:#ef4444">No files found for this job.</small>'
            return
        import tempfile as _tf
        local_dl_dir = _tf.mkdtemp(prefix='hbv_dl_')
        count = 0
        errors = []
        for sid, file_list in remote_files.items():
            sid_dir = os.path.join(local_dl_dir, sid)
            os.makedirs(sid_dir, exist_ok=True)
            for hpc_path in file_list:
                fname = os.path.basename(hpc_path)
                try:
                    csv_bytes = _api.download_csv(job_id, sid, fname)
                    with open(os.path.join(sid_dir, fname), 'wb') as _f:
                        _f.write(csv_bytes)
                    count += 1
                except Exception as e:
                    errors.append(f'{sid}/{fname}: {e}')
        local_results = {
            sid: [os.path.join(local_dl_dir, sid, os.path.basename(p)) for p in paths]
            for sid, paths in remote_files.items()
        }
        flat = {f'{sid}_{os.path.basename(p)}': p
                for sid, paths in local_results.items() for p in paths if p.endswith('.csv')}
        file_dd.options = flat
        state['output_files'] = {os.path.basename(p): p
                                  for paths in local_results.values() for p in paths}
        if errors:
            log(f'Download errors: {errors}', 'warn')
        threading.Thread(target=_save_to_workspace, args=(job_id, local_results), daemon=True).start()
        log(f'Loaded {count} file(s) from job {job_id[:8]} — see Output tab', 'ok')
        _dl_job_lbl.value = f'<small style="color:#4ade80">✔ {count} file(s) loaded — saved to ~/hbv_results/{job_id[:8]}/</small>'
        right_tabs.selected_index = 1
    except Exception as exc:
        log(f'Download job {job_id[:8]} error: {exc}', 'error')
        _dl_job_lbl.value = f'<small style="color:#ef4444">Error: {exc}</small>'

right_jobs_tab = widgets.VBox([
    widgets.HBox([
        widgets.HTML('<div style="font-size:13px;font-weight:700;color:#e2e8f0">My Jobs</div>'),
        _jobs_refresh_btn, _jobs_status,
    ], layout=widgets.Layout(align_items='center', gap='8px', margin_bottom='8px')),
    _jobs_rows_box,
    _dl_job_lbl,
], layout=widgets.Layout(padding='12px', background_color=_BG2, overflow_y='auto'))

# ── System / Monitor tab ─────────────────────────────────────────────────
_sys_html    = widgets.HTML('<div style="color:#64748b;font-size:12px;padding:8px">Click Refresh to load.</div>')
_sys_refresh = widgets.Button(description='⟳ Refresh', button_style='', icon='refresh',
                               layout=widgets.Layout(width='120px'))
_cancel_btn  = widgets.Button(description='✕ Cancel Job', button_style='danger', icon='times',
                               layout=widgets.Layout(width='140px', display='none'))
_res_html    = widgets.HTML('')   # live resource panel shown during run

_sys_cache = {'ts': 0, 'stats': None}

def _render_system():
    import time as _time
    _api_start()
    _sys_html.value = '<div style="color:#64748b;font-size:12px;padding:8px">Loading…</div>'
    try:
        health  = _api.get_cluster_info()
        # Cache disk stats for 60s — du on NFS is slow
        if _time.time() - _sys_cache['ts'] > 60 or _sys_cache['stats'] is None:
            _sys_cache['stats'] = _api.get_system_stats()
            _sys_cache['ts'] = _time.time()
        stats = _sys_cache['stats']
        s       = health.get('summary', {})
        total_c = s.get('total_cpus', '—')
        idle_c  = s.get('idle_cpus', '—')
        alloc_c = s.get('alloc_cpus', '—')
        nodes   = health.get('nodes', [])
        node_rows = ''.join(
            f'<tr><td style="padding:2px 8px">{n["name"]}</td>'
            f'<td style="padding:2px 8px">{n["state"]}</td>'
            f'<td style="padding:2px 8px">{n["cpus_alloc"]}/{n["cpus_total"]}</td>'
            f'<td style="padding:2px 8px">{n["mem_gb"]} GB</td></tr>'
            for n in nodes
        )
        fs_free  = stats.get('fs_free_gb',  '—')
        fs_total = stats.get('fs_total_gb', '—')
        u_out    = stats.get('user_output_usage', '—')
        nfs_out  = stats.get('nfs_output', '—')
        nfs_upl  = stats.get('nfs_uploads', '—')

        # Check for any active running job and show live resource block
        active_block = ''
        try:
            jobs = _api.my_jobs()
            running = [j for j in jobs if j['status'] in ('running', 'queued')]
            if running:
                j = running[0]
                jid = j['job_id']
                n_catches = len(j.get('catchments', []))
                res = _api.get_job_resources(jid)
                nodes_list = res.get('nodes_list', [])
                if not nodes_list:
                    n_str = res.get('nodes', '—')
                    c_str = res.get('cpus', '—')
                    nodes_list = [{'name': n_str, 'cpus': c_str}] if n_str and n_str != '—' else []
                mem_avg = res.get('mem_avg_kb', '')
                mem_str = ''
                if mem_avg and mem_avg != '—':
                    try:
                        mem_str = f'&nbsp;💾 RAM avg: <b>{int(mem_avg)//1024} MB</b>'
                    except Exception:
                        pass
                node_lines = ''.join(
                    f'🖥 <b style="color:#e2e8f0">{nd["name"]}</b>'
                    f' &nbsp;⚙ <b style="color:#e2e8f0">{nd["cpus"]} CPUs</b><br>'
                    for nd in nodes_list
                ) or '🖥 —<br>'
                status_color = '#60a5fa' if j['status'] == 'running' else '#f59e0b'
                active_block = f"""
  <br><b style="color:#94a3b8">ACTIVE JOB</b>
  <div style="background:#1e293b;border-radius:6px;padding:6px 10px;margin-top:4px;line-height:1.8">
    <span style="color:{status_color}">⬤</span>
    <b style="color:#e2e8f0">{j['status'].capitalize()}</b>
    &nbsp;·&nbsp; Job <code style="color:#94a3b8">{jid[:8]}</code>
    &nbsp;·&nbsp; {n_catches} catchment(s)<br>
    {node_lines}
    ⚡ <b style="color:#e2e8f0">{n_catches}</b> catchments running in parallel{mem_str}
  </div>"""
        except Exception:
            pass

        try:
            _quota_num = float(''.join(c for c in str(u_out) if c.isdigit() or c == '.') or '0')
            # convert to GB if unit present
            if 'M' in str(u_out): _quota_num /= 1024
            elif 'K' in str(u_out): _quota_num /= 1024**2
            elif 'T' in str(u_out): _quota_num *= 1024
            _quota_pct = (_quota_num / 100) * 100
        except Exception:
            _quota_pct = 0
        _sys_html.value = f"""
<div style="font-size:12px;padding:8px;line-height:1.8">
  <b style="color:#94a3b8">CLUSTER</b><br>
  CPUs: <b>{alloc_c}</b> allocated / <b>{idle_c}</b> idle / <b>{total_c}</b> total<br>
  <div style="max-height:160px;overflow-y:auto;margin-top:4px"><table style="border-collapse:collapse;width:100%">
    <tr style="color:#64748b;position:sticky;top:0;background:#0f172a"><th style="padding:2px 8px;text-align:left">Node</th>
      <th>State</th><th>CPUs used/total</th><th>RAM</th></tr>
    {node_rows or '<tr><td colspan=4 style="color:#64748b">No node data</td></tr>'}
  </table></div>
  {active_block}
  <br><b style="color:#94a3b8">DISK</b><br>
  NFS free: <b>{fs_free} GB</b> / {fs_total} GB total<br>
  All output: {nfs_out} &nbsp;|&nbsp; Uploads: {nfs_upl}<br>
  Your quota: <b>{u_out}</b> / 100 GB
  <div style="background:#334155;border-radius:4px;height:6px;margin:4px 0 0 0;width:100%">
    <div style="background:{'#ef4444' if _quota_pct>90 else '#f59e0b' if _quota_pct>70 else '#4ade80'};height:6px;border-radius:4px;width:{min(_quota_pct,100):.0f}%"></div>
  </div>
</div>"""
    except Exception as exc:
        _sys_html.value = f'<div style="color:#ef4444;padding:8px">Error: {exc}</div>'
    finally:
        _api_done()

def _on_sys_refresh(b):
    threading.Thread(target=_render_system).start()
_sys_refresh.on_click(_on_sys_refresh)

right_sys_tab = widgets.VBox([
    widgets.HBox([_sys_refresh], layout=widgets.Layout(padding='8px 12px')),
    _sys_html,
], layout=widgets.Layout(padding='4px', background_color=_BG2, overflow_y='auto'))

right_tabs = widgets.Tab(children=[right_log_tab, right_output_tab, right_jobs_tab, right_sys_tab],
                          layout=widgets.Layout(width='100%'))
right_tabs.set_title(0, '📋 Logs')
right_tabs.set_title(1, '📊 Output')
right_tabs.set_title(2, '📁 My Jobs')
right_tabs.set_title(3, '🖥 System')

# Auto-load job history / system stats when tabs opened
def _on_right_tab(change):
    if change['new'] == 2:
        threading.Thread(target=_render_jobs).start()
    elif change['new'] == 3:
        threading.Thread(target=_render_system).start()
right_tabs.observe(_on_right_tab, names='selected_index')

panel_w_slider = widgets.IntSlider(value=400, min=300, max=800, step=10, description='w:',
                                    style={'description_width':'20px'}, layout=widgets.Layout(width='200px'))
panel_w_slider.observe(lambda c: setattr(right_panel_vbox.layout, 'width', f'{c["new"]}px'), names='value')

# ═══════════════════════════════════════════════════════════════════════════
# ── Custom Catchment Tab ───────────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════
from dashboard_utils.custom_catchment import build_tab as _build_catchment_tab
_cc_refs = {}
custom_catchment_tab = _build_catchment_tab(state, _cc_refs)

# ═══════════════════════════════════════════════════════════════════════════
# ── Build step panels ──────────────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════
def _card(title, *children, flex='1', min_width='0'):
    header = widgets.HTML(
        f'<div style="font-size:12px;font-weight:700;color:#94a3b8;'
        f'padding-bottom:8px;border-bottom:1px solid {_BDR};margin-bottom:10px">{title}</div>'
    )
    return widgets.VBox(
        [header, *children],
        layout=widgets.Layout(flex=flex, min_width=min_width,
                               border=f'1px solid {_BDR}', border_radius='10px',
                               padding='14px', margin='0',
                               background_color=_BG2),
    )

# ── Step 1 ─────────────────────────────────────────────────────────────────
step1_sidebar = widgets.VBox([
    section_lbl('CATCHMENT LEVEL — TASO'),
    widgets.HBox([taso_toggle, taso_lbl], layout=widgets.Layout(align_items='center', gap='8px')),
    divider(),
    section_lbl('SHAPEFILE SOURCE'),
    shp_source_toggle,
    widgets.HBox([shp_load_btn, shp_auto_prog, shp_auto_lbl],
                  layout=widgets.Layout(align_items='center', gap='6px', margin='6px 0')),
    upload_shp,
    widgets.HBox([upload_shp_load_btn, upload_shp_lbl], layout=widgets.Layout(align_items='center', gap='6px')),
    shp_pick_lbl, divider(),
    section_lbl('SEARCH CATCHMENTS'),
    search_unified, search_count_lbl, _chip_trigger, _chips_html, divider(),
    section_lbl('SELECTED'),
    _sel_count_lbl, _sel_tags_html,
    widgets.HBox([clear_sel_btn, _random_n_inp, _random_sel_btn], layout=widgets.Layout(margin='4px 0', align_items='center', gap='6px')),
], layout=widgets.Layout(
    width='310px', min_width='280px', flex_shrink='0',
    overflow_y='auto', height=_STEP1_H, min_height='300px',
    padding='14px', border_right=f'1px solid {_BDR2}',
    background_color=_BG2,
))

step1_map_pane = widgets.VBox([
    map_container,
    widgets.HBox([viewport_lbl], layout=widgets.Layout(
        padding='3px 8px', border_top=f'1px solid {_BDR2}', background_color=_BG1)),
    catchment_status,
], layout=widgets.Layout(flex='1 1 auto', min_width='0', height=_STEP1_H, min_height='300px',
                          background_color=_BG1))

step1_panel = widgets.HBox(
    [step1_sidebar, step1_map_pane],
    layout=widgets.Layout(height=_STEP1_H, min_height='300px', background_color=_BG1),
)

# ── Step 2 ─────────────────────────────────────────────────────────────────
step2_left = _card('📅  Dates &amp; Storage',
    section_lbl('SIMULATION PERIOD'),
    widgets.VBox([date_start, date_end, date_lbl]),
    divider(), section_lbl('DOWNLOAD FOLDER'),
    download_dir_input, _fb_status,
    widgets.HBox([_fb_up_btn, _fb_new_name, _fb_select_btn],
                  layout=widgets.Layout(gap='6px', align_items='center', margin='6px 0')),
    _fb_listing, _fb_click_input,
    flex='0 0 340px', min_width='300px',
)
climate_tabs = widgets.Tab(children=[precip_panel, et_panel, temp_panel],
                            layout=widgets.Layout(width='100%'))
climate_tabs.set_title(0, '🌧 Precipitation')
climate_tabs.set_title(1, '💧 Evapotranspiration')
climate_tabs.set_title(2, '🌡 Temperature')
step2_right = _card('🌍  Climate Data',
    tip('Select source for each variable. "Copy dates" syncs with global dates above.'),
    climate_tabs, flex='1', min_width='0')
step2_panel = widgets.HBox([step2_left, step2_right],
    layout=widgets.Layout(gap='14px', padding='14px', align_items='flex-start', background_color=_BG1))

# ── Step 3 ─────────────────────────────────────────────────────────────────
step3_panel = widgets.HBox(
    [_card('🏙  Urban Land',
            tip('Shapefile of urban/built-up areas for HBV impervious fraction.'),
            urban_panel, flex='1', min_width='0'),
     _card('🌾  Agricultural Land',
            tip('Shapefile of agricultural areas for HBV soil parameter weighting.'),
            agri_panel, flex='1', min_width='0')],
    layout=widgets.Layout(gap='14px', padding='14px', align_items='flex-start', background_color=_BG1))

# ── Step 4 ─────────────────────────────────────────────────────────────────
step4_panel = widgets.HBox(
    [_card('⚙  Parameters &amp; Compute',
            section_lbl('HBV PARAMETER FILE'),
            widgets.HBox([upload_hbvpara, lbl_hbvpara],
                          layout=widgets.Layout(align_items='center', gap='10px', margin='4px 0')),
            divider(), section_lbl('OUTPUT TEMPORAL RESOLUTION'), output_freq, divider(),
            section_lbl('MPI / CPU CORES'), mpi_status_lbl,
            widgets.HBox([cluster_info_lbl, cluster_refresh_btn],
                          layout=widgets.Layout(align_items='center', gap='8px', margin='4px 0')),
            widgets.HBox([cpu_slider],
                          layout=widgets.Layout(align_items='center', gap='8px', margin='2px 0')),
            widgets.HBox([n_nodes_slider],
                          layout=widgets.Layout(align_items='center', gap='8px', margin='2px 0')),
            widgets.HBox([partition_dd],
                          layout=widgets.Layout(align_items='center', gap='8px', margin='2px 0')),
            tip('CPUs/node × # nodes = total parallelism. Each node runs one Slurm task. Catchments split across tasks.'),
            divider(), section_lbl('OUTPUT FOLDER'), project_dir_input, divider(),
            section_lbl('ADVANCED — PREDATA.CSV OVERRIDE'), fallback_toggle, fallback_box,
            flex='0 0 360px', min_width='300px'),
     _card('🚀  Pre-flight &amp; Run',
            tip('All items below must be ✅ before running.'),
            readiness_html, divider(), widgets.HBox([run_btn, _cancel_btn]), _res_html, run_lbl,
            flex='1', min_width='0')],
    layout=widgets.Layout(gap='14px', padding='14px', align_items='flex-start', background_color=_BG1))

# ═══════════════════════════════════════════════════════════════════════════
# ── Stepper navigation ─────────────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════
stepper_html = widgets.HTML(render_stepper(0))
_cur_step    = [0]
_all_panels  = [step1_panel, step2_panel, step3_panel, step4_panel]
_step_labels = ['Catchment', 'Climate', 'Land Use', 'Run']

back_btn    = widgets.Button(description='← Back',  layout=widgets.Layout(width='100px'), disabled=True)
next_btn    = widgets.Button(description='Next →',   layout=widgets.Layout(width='140px'), button_style='info')
step_status = widgets.HTML('')

def _goto(n):
    n = max(0, min(3, n)); _cur_step[0] = n
    for i, p in enumerate(_all_panels): p.layout.display = '' if i == n else 'none'
    stepper_html.value = render_stepper(n)
    back_btn.disabled  = (n == 0)
    if n == 3:
        next_btn.description = '🚀 Run'; next_btn.button_style = 'success'; _update_readiness()
    else:
        next_btn.description = 'Next →'; next_btn.button_style = 'info'
    step_status.value = f'<span style="font-size:12px;color:#475569">Step {n+1} of 4 — {_step_labels[n]}</span>'

back_btn.on_click(lambda b: _goto(_cur_step[0] - 1))
next_btn.on_click(lambda b: run_model(b) if _cur_step[0] == 3 else _goto(_cur_step[0] + 1))

nav_bar = widgets.HBox(
    [back_btn, step_status, next_btn],
    layout=widgets.Layout(justify_content='space-between', align_items='center',
                           padding='9px 14px', border_top=f'1px solid {_BDR2}',
                           background_color=_BG1),
)

# ── Assemble left panel ────────────────────────────────────────────────────
step_content = widgets.VBox(_all_panels + [nav_bar],
                             layout=widgets.Layout(flex='1 1 auto', background_color=_BG1))

left_tabs = widgets.Tab(
    children=[
        widgets.VBox([stepper_html, step_content],
                      layout=widgets.Layout(background_color=_BG1)),
        custom_catchment_tab,
    ],
    layout=widgets.Layout(flex='1 1 auto'),
)
left_tabs.set_title(0, '⚙  HBV Workflow')
left_tabs.set_title(1, '🗺  Custom Catchment')
_cc_refs['go_to_input'] = lambda: setattr(left_tabs, 'selected_index', 0)

left_panel_vbox = widgets.VBox(
    [app_header_widget(), left_tabs],
    layout=widgets.Layout(flex='1 1 auto', min_width='0',
                           height=_PNL_H, min_height='400px',
                           overflow_y='hidden',
                           background_color=_BG1,
                           border=f'1px solid {_BDR2}', border_radius='10px'),
)

right_panel_vbox = widgets.VBox(
    [right_panel_header(),
     widgets.HBox(
         [widgets.HTML(f'<small style="color:{_BDR}">Panel width:</small>'), panel_w_slider],
         layout=widgets.Layout(align_items='center', gap='8px', padding='5px 12px',
                                border_bottom=f'1px solid {_BDR2}', background_color=_BG0),
     ),
     right_tabs],
    layout=widgets.Layout(width='400px', min_width='280px', flex_shrink='0',
                           height=_PNL_H, min_height='400px',
                           border=f'1px solid {_BDR2}', border_radius='10px',
                           overflow='hidden', background_color=_BG0),
)

root = widgets.HBox(
    [left_panel_vbox, right_panel_vbox],
    layout=widgets.Layout(width='100%', align_items='flex-start',
                           gap='12px', padding='14px', background_color=_BG0),
)

_goto(0)
display(root)

# ═══════════════════════════════════════════════════════════════════════════
# Validation + Run
# ═══════════════════════════════════════════════════════════════════════════
def _validate():
    if fallback_toggle.value:
        return (None, 'Upload predata.csv or disable override') if not state.get('predata_path') else (None, None)
    errs = []
    if not state.get('selected_ids'):               errs.append('select catchment (Step 1)')
    if not date_start.value or not date_end.value:  errs.append('set dates (Step 2)')
    elif date_end.value <= date_start.value:         errs.append('end date after start')
    if not state.get('precipitation_nc'):           errs.append('Precipitation')
    if not state.get('evapotranspiration_nc'):      errs.append('ET')
    if not state.get('temperature_nc'):             errs.append('Temperature')
    if not state.get('urban_land_path'):            errs.append('Urban land')
    if not state.get('agricultural_land_path'):     errs.append('Agricultural land')
    if not state.get('hbvpara_path'):               errs.append('hbv_para.csv')
    if errs: return None, 'Missing: ' + ' | '.join(errs)
    return {
        'shapefile_path':                  state['shapefile_path'],
        'catchment_id_name':               state['id_col'],
        'precipitation_nc_file_path':      state['precipitation_nc'],
        'evapotranspiration_nc_file_path': state['evapotranspiration_nc'],
        'temperature_nc_file_path':        state['temperature_nc'],
        'output_csv_path':                 os.path.join(state['temp_dir'], 'met_input.csv'),
        'urban_land_path':                 state['urban_land_path'],
        'agricultural_land_path':          state['agricultural_land_path'],
        'csv_parameters_path':             state['hbvpara_path'],
        'start_date':                      str(date_start.value),
        'end_date':                        str(date_end.value),
        'output_frequency':                output_freq.value,
    }, None

def run_model(b):
    cfg, err = _validate()
    if err:
        run_lbl.value = f'<small style="color:#ef4444">{err}</small>'
        log(f'Validation: {err}', 'error'); return
    run_btn.disabled = True
    run_lbl.value = f'<small style="color:{_BDR}">Starting…</small>'
    res_out.clear_output(); logging_utils.clear_log(); log('─'*40, 'info'); log('HBV run starting', 'start')

    if _USE_API:
        # ── HPC path: submit via FastAPI ────────────────────────────────
        def _run_api():
            try:
                # ── Upload all input files to HPC NFS first ───────────────
                def _upload(local_path, label):
                    if not local_path:
                        return ''
                    # Already an NFS/HPC path — pass through directly
                    if local_path.startswith('/data/hbv/') or local_path.startswith('/mnt/'):
                        log(f'Using HPC path for {label}', 'info')
                        return local_path
                    if not os.path.exists(local_path):
                        return local_path
                    size_mb = os.path.getsize(local_path) / 1024 / 1024
                    run_lbl.value = f'<small style="color:{_BDR}">Uploading {label} ({size_mb:.0f} MB)…</small>'
                    log(f'Uploading {label} ({size_mb:.1f} MB)…', 'info')
                    def _prog(sent, total):
                        pct = int(sent / total * 100)
                        run_lbl.value = f'<small style="color:{_BDR}">{label}: {pct}%</small>'
                    if local_path.lower().endswith('.shp'):
                        return _api.upload_shapefile_dir(local_path, progress_cb=_prog)
                    return _api.upload_file(local_path, progress_cb=_prog)

                # For shapefile: check if HPC already has this Taso registered — skip upload if so
                shp_nfs = ''
                taso_key = state.get('shapefile_taso_key')
                if taso_key:
                    try:
                        entries = _api.get_shapefiles()
                        hpc_entry = next((e for e in entries if e['key'] == taso_key and e['exists']), None)
                        if hpc_entry:
                            log(f'Using HPC pre-loaded shapefile for {taso_key}', 'info')
                            shp_nfs = hpc_entry['path']
                    except Exception:
                        pass
                if not shp_nfs:
                    shp_nfs = _upload(state.get('shapefile_path'), 'shapefile')
                precip_nfs = _upload(state.get('precipitation_nc'),       'precipitation NC')
                et_nfs     = _upload(state.get('evapotranspiration_nc'),  'ET NC')
                temp_nfs   = _upload(state.get('temperature_nc'),         'temperature NC')
                urban_nfs  = _upload(state.get('urban_land_path'),        'urban land')
                agri_nfs   = _upload(state.get('agricultural_land_path'), 'agri land')
                para_nfs   = _upload(state.get('hbvpara_path'),          'HBV parameters')

                run_lbl.value = f'<small style="color:{_BDR}">Submitting job…</small>'
                log('All files uploaded — submitting job', 'info')

                job = _api.submit_job(
                    catchment_ids          = sorted(state['selected_ids'], key=str),
                    shapefile_path         = shp_nfs,
                    id_col                 = state.get('id_col', 'TASO_ID'),
                    precipitation_nc       = precip_nfs,
                    evapotranspiration_nc  = et_nfs,
                    temperature_nc         = temp_nfs,
                    urban_land_path        = urban_nfs,
                    agricultural_land_path = agri_nfs,
                    hbvpara_path           = para_nfs,
                    n_nodes                = n_nodes_slider.value,
                    cpus_per_task          = cpu_slider.value,
                    partition              = partition_dd.value,
                )
                job_id = job['job_id']
                log(f'Job submitted → {job_id}', 'info')
                run_lbl.value = f'<small style="color:{_BDR}">Queued: {job_id[:8]}…</small>'

                # Show cancel button while job is active
                _cancel_btn.layout.display = ''
                _cancel_btn.description = '✕ Cancel Job'
                _cancel_btn.disabled = False
                _cancel_requested = [False]
                _cancel_job_id = [job_id]
                def _on_cancel(b):
                    _cancel_btn.disabled = True
                    _cancel_btn.description = '⏳ Cancelling…'
                    run_lbl.value = '<small style="color:#f59e0b">⏳ Cancelling job…</small>'
                    _cancel_requested[0] = True
                    def _do_cancel():
                        try:
                            # Check current status before attempting cancel
                            current = _api.get_status(_cancel_job_id[0])
                            if current['status'] in ('done', 'failed', 'cancelled'):
                                return  # job already finished, poll loop handles everything
                            _api.cancel_job(_cancel_job_id[0])
                            run_lbl.value = '<small style="color:#f59e0b">Cancelled</small>'
                            def _reset():
                                import time as _t; _t.sleep(3)
                                run_lbl.value = f'<small style="color:#94a3b8">Ready</small>'
                            threading.Thread(target=_reset, daemon=True).start()
                        except Exception as ex:
                            if not (hasattr(ex, 'response') and ex.response is not None and ex.response.status_code == 409):
                                run_lbl.value = f'<small style="color:#ef4444">Cancel failed: {ex}</small>'
                        finally:
                            _cancel_btn.layout.display = 'none'
                            _res_html.value = ''
                            run_btn.disabled = False
                    threading.Thread(target=_do_cancel, daemon=True).start()
                _cancel_btn.on_click(_on_cancel)

                _log_offset = 0

                def _tick(status):
                    nonlocal _log_offset
                    if _cancel_requested[0]:
                        return  # don't overwrite the "Cancelling…" label
                    s = status['status']
                    n = len(status.get('catchments', []))
                    if s == 'running':
                        colour = '#60a5fa'
                        label  = f'Running ({n} catchments)…'
                    elif s == 'queued':
                        colour = '#f59e0b'
                        label  = '⏳ Queued — waiting for cluster resources…'
                    else:
                        colour = _BDR
                        label  = f'{s.capitalize()} ({n} catchments)…'
                    run_lbl.value = f'<small style="color:{colour}">{label}</small>'
                    # Live resource stats — fetch when running, keep last value when done
                    try:
                        if s in ('running', 'queued'):
                            res       = _api.get_job_resources(job_id)
                            mem_avg   = res.get('mem_avg_kb', '')
                            mem_str   = ''
                            if mem_avg and str(mem_avg) not in ('', '—'):
                                try:
                                    mem_str = f'&nbsp;💾 <b style="color:#e2e8f0">{int(mem_avg)//1024} MB RAM avg</b>'
                                except Exception:
                                    pass
                            nodes_list = res.get('nodes_list', [])
                            if not nodes_list:
                                # fallback: single line
                                nodes_str = res.get('nodes', '—')
                                cpus      = res.get('cpus', '—')
                                if nodes_str and nodes_str != '—':
                                    nodes_list = [{'name': nodes_str, 'cpus': cpus}]
                            if nodes_list:
                                node_lines = ''.join(
                                    f'🖥 <b style="color:#e2e8f0">{nd["name"]}</b>'
                                    f' &nbsp;⚙ <b style="color:#e2e8f0">{nd["cpus"]} CPUs</b><br>'
                                    for nd in nodes_list
                                )
                                _res_html.value = (
                                    f'<div style="font-size:11px;color:#94a3b8;padding:2px 0;line-height:1.8">'
                                    f'{node_lines}'
                                    f'⚡ <b style="color:#e2e8f0">{n}</b> catchments running in parallel'
                                    f'&nbsp;{mem_str}</div>'
                                )
                    except Exception:
                        pass
                    # Fetch new log lines since last poll
                    try:
                        log_data = _api.get_logs(job_id, offset=_log_offset)
                        _log_offset = log_data['next_offset']
                        for line in log_data['lines']:
                            kind = ('error' if 'ERROR' in line or 'error' in line.lower()
                                    else 'warn'  if 'WARNING' in line or 'WARN' in line
                                    else 'ok'    if 'DONE' in line or 'done' in line.lower()
                                    else 'run'   if any(x in line for x in ['[HBV]','[rank','[LOCAL','[SLURM'])
                                    else 'info')
                            log(line, kind)
                    except Exception:
                        pass  # log fetch is best-effort

                # Poll loop — exits immediately when cancel is requested
                import time as _time
                final = None
                while True:
                    status = _api.get_status(job_id)
                    if _cancel_requested[0]:
                        final = status; break
                    _tick(status)
                    if status['status'] in ('done', 'failed', 'cancelled'):
                        _cancel_btn.disabled = True
                        _cancel_btn.layout.display = 'none'
                        final = status; break
                    _time.sleep(5)
                if _cancel_requested[0]:
                    return

                # Fetch any remaining logs after job finishes
                try:
                    log_data = _api.get_logs(job_id, offset=_log_offset)
                    for line in log_data['lines']:
                        kind = ('error' if 'ERROR' in line or 'Traceback' in line
                                else 'ok' if 'done' in line.lower() or 'succeeded' in line
                                else 'info')
                        log(line, kind)
                except Exception:
                    pass

                if final['status'] == 'cancelled':
                    log('Job cancelled', 'warn')
                    return
                if final['status'] != 'done':
                    msg = final.get('error_msg') or final['status']
                    run_lbl.value = f'<small style="color:#ef4444">{msg}</small>'
                    log(f'Job {final["status"]}: {msg}', 'error'); return

                res = _api.get_results(job_id)
                remote_files = res.get('files', {})  # {catchment: [hpc_path, ...]}

                # Download all CSVs from API into a local temp dir
                import tempfile as _tf
                local_dl_dir = _tf.mkdtemp(prefix='hbv_results_')
                local_results = {}  # {catchment: [local_path, ...]}
                for sid, file_list in remote_files.items():
                    sid_dir = os.path.join(local_dl_dir, sid)
                    os.makedirs(sid_dir, exist_ok=True)
                    local_paths = []
                    for hpc_path in file_list:
                        fname = os.path.basename(hpc_path)
                        local_path = os.path.join(sid_dir, fname)
                        try:
                            csv_bytes = _api.download_csv(job_id, sid, fname)
                            with open(local_path, 'wb') as _f: _f.write(csv_bytes)
                            local_paths.append(local_path)
                        except Exception as ex:
                            log(f'Download {fname}: {ex}', 'warn')
                    local_results[sid] = local_paths

                freq = output_freq.value
                if freq != 'daily':
                    freq_map = {'weekly':'W','monthly':'ME','yearly':'YE'}
                    for sid, file_list in list(local_results.items()):
                        agg_list = []
                        for path in file_list:
                            try:
                                df = pd.read_csv(path)
                                date_cols = [c for c in df.columns if any(x in c.lower() for x in ['date','time'])]
                                if date_cols:
                                    df['_dt'] = pd.to_datetime(df[date_cols].astype(str).agg('-'.join,axis=1),errors='coerce')
                                    df = df.set_index('_dt').select_dtypes(include='number').resample(freq_map[freq]).mean().reset_index()
                                    agg_path = path.replace('.csv',f'_{freq}.csv'); df.to_csv(agg_path,index=False)
                                    agg_list.append(agg_path)
                            except Exception as ex: log(f'Agg {path}: {ex}','warn')
                        if agg_list: local_results[sid] = file_list + agg_list

                flat = {f'{sid}_{os.path.basename(p)}': p
                        for sid, paths in local_results.items() for p in paths if p.endswith('.csv')}
                file_dd.options = flat; right_tabs.selected_index = 1
                n_ok = len(local_results)
                run_lbl.value = f'<small style="color:#4ade80">✔ Done — {n_ok} catchment(s)</small>'
                state['output_files'] = {os.path.basename(p): p
                                          for paths in local_results.values() for p in paths}
                log(f'Complete — {n_ok} catchment(s) done','done')
                threading.Thread(target=_save_to_workspace, args=(job_id, local_results), daemon=True).start()

            except Exception as exc:
                import traceback as _tb
                if hasattr(exc, 'response') and exc.response is not None and exc.response.status_code == 429:
                    msg = exc.response.json().get('detail', 'Too many active jobs')
                    run_lbl.value = f'<small style="color:#f59e0b">⏳ {msg}</small>'
                    log(msg, 'warn')
                elif hasattr(exc, 'response') and exc.response is not None and exc.response.status_code == 507:
                    msg = exc.response.json().get('detail', 'Storage quota exceeded')
                    run_lbl.value = f'<small style="color:#ef4444">💾 {msg}</small>'
                    log(msg, 'error')
                else:
                    run_lbl.value = f'<small style="color:#ef4444">{exc}</small>'
                    log(_tb.format_exc(), 'error')
            finally:
                run_btn.disabled = False
                _cancel_btn.layout.display = 'none'
                _res_html.value = ''
        threading.Thread(target=_run_api, daemon=True).start()

    else:
        # ── Local path: call hbv_worker directly (dev / single-machine) ─
        def _run():
            try:
                n_catches = len(state['selected_ids']); n_ranks = min(cpu_slider.value, max(n_catches, 1))
                run_tmp  = tempfile.mkdtemp(prefix='hbv_mpi_')
                cfg_path = os.path.join(run_tmp, 'run_config.json')
                res_path = os.path.join(run_tmp, 'results.json')
                worker_cfg = {
                    'catchment_ids':          sorted(state['selected_ids'], key=str),
                    'shapefile_path':         state['shapefile_path'] or '',
                    'id_col':                 state['id_col'],
                    'precipitation_nc':       state.get('precipitation_nc',''),
                    'evapotranspiration_nc':  state.get('evapotranspiration_nc',''),
                    'temperature_nc':         state.get('temperature_nc',''),
                    'urban_land_path':        state.get('urban_land_path',''),
                    'agricultural_land_path': state.get('agricultural_land_path',''),
                    'hbvpara_path':           state['hbvpara_path'],
                    'out_root':               project_dir_input.value.strip() or os.path.join(run_tmp,'outputs'),
                    'temp_dir':               run_tmp,
                }
                with open(cfg_path,'w') as f: json.dump(worker_cfg, f, indent=2)
                log(f'Config: {cfg_path}','info')
                cmd = ([_MPIRUN,'-n',str(n_ranks),sys.executable,_WORKER_SCRIPT,'--config',cfg_path,'--results',res_path]
                       if _MPIRUN else [sys.executable,_WORKER_SCRIPT,'--config',cfg_path,'--results',res_path])
                log(f'CMD: {" ".join(cmd)}','info')
                env = os.environ.copy(); env['HYDRA_BOOTSTRAP'] = 'fork'
                proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
                for line in proc.stdout:
                    line = line.rstrip()
                    if not line: continue
                    kind = ('error' if 'ERROR' in line else 'warn' if 'WARNING' in line
                            else 'ok' if 'DONE' in line or '✅' in line
                            else 'run' if '[HBV]' in line else 'info')
                    log(line, kind); run_lbl.value = f'<small style="color:{_BDR}">{line[:120]}</small>'
                proc.wait()
                if proc.returncode != 0:
                    run_lbl.value = f'<small style="color:#ef4444">Exit {proc.returncode}</small>'
                    log(f'Failed (exit {proc.returncode})','error'); return
                if not os.path.exists(res_path):
                    run_lbl.value = '<small style="color:#ef4444">No results.json</small>'; return
                with open(res_path) as f: payload = json.load(f)
                results = payload.get('results',{}); errors = payload.get('errors',{})
                for sid,tb in errors.items(): log(f'Catchment {sid} FAILED: {tb}','error')
                freq = output_freq.value
                if freq != 'daily':
                    freq_map = {'weekly':'W','monthly':'ME','yearly':'YE'}
                    for sid,files in results.items():
                        for k,path in list(files.items()):
                            try:
                                df = pd.read_csv(path)
                                date_cols = [c for c in df.columns if any(x in c.lower() for x in ['date','time'])]
                                if date_cols:
                                    df['_dt'] = pd.to_datetime(df[date_cols].astype(str).agg('-'.join,axis=1),errors='coerce')
                                    df = df.set_index('_dt').select_dtypes(include='number').resample(freq_map[freq]).mean().reset_index()
                                    agg_path = path.replace('.csv',f'_{freq}.csv'); df.to_csv(agg_path,index=False)
                                    results[sid][f'{k}_{freq}'] = agg_path
                            except Exception as e: log(f'Agg {k}: {e}','warn')
                flat = {f'{sid}_{k}': p for sid,files in results.items() for k,p in files.items()}
                file_dd.options = flat; right_tabs.selected_index = 1
                n_ok = len(results); n_err = len(errors)
                run_lbl.value = f'<small style="color:#4ade80">✔ Done — {n_ok} OK{f", {n_err} failed" if n_err else ""}</small>'
                state['output_files'] = next(iter(results.values()),{})
                log(f'Complete — {n_ok} succeeded, {n_err} failed','done')
            except Exception as exc:
                import traceback
                run_lbl.value = f'<small style="color:#ef4444">{exc}</small>'
                log(traceback.format_exc(),'error')
            finally:
                run_btn.disabled = False
        threading.Thread(target=_run, daemon=True).start()
run_btn.on_click(run_model)

file_dd.observe(
    lambda c: setattr(col_dd,'options',
        pd.read_csv(c['new']).select_dtypes(include='number').columns.tolist() if c['new'] else []),
    names='value')

def on_plot(b):
    res_out.clear_output()
    with res_out:
        try:
            df = pd.read_csv(file_dd.value); col = col_dd.value
            val = df[col].mean() if agg_tog.value == 'mean' else df[col].sum()
            print(f'{agg_tog.value.title()} of "{col}": {val:.4f}')
            gdf = state['gdf'] or gpd.read_file(state['shapefile_path'])
            gdf['plot_value'] = 0.0
            gdf.loc[gdf[state['id_col']] == state['selected_id'], 'plot_value'] = val
            fig, ax = plt.subplots(figsize=(8,6))
            gdf.plot(ax=ax, column='plot_value', cmap='Blues', legend=True,
                     edgecolor='black', missing_kwds={'color':'lightgrey'})
            ax.set_title(f'{agg_tog.value.title()} of {col} — {state["selected_id"]}')
            plt.tight_layout(); plt.show()
        except Exception as e: print(f'Error: {e}')
plot_btn.on_click(on_plot)

log(f'HBV Dashboard ready | Python {sys.version.split()[0]} | CPUs: {_max_cpu} | mpirun: {_MPIRUN or "NOT FOUND"}', 'ok')
